# Oracle Enterprise Workflow in JupyterLab (`telecom_churn`)

This notebook is a **production-style simulation** for connecting to an enterprise Oracle database from JupyterLab, extracting `telecom_churn`, and extending SQL output with Python analytics/visualization.

**Connection profile provided:** `DIPRODU` (host `10.213.207.21`, port `1521`, service `stdiprodu.internal.altconsult.com`, user `MANAMUSA`).

## 0) Why this workflow is enterprise-safe

- Uses **environment variables first** for credentials.
- Falls back to **interactive password prompt** if password env var is missing.
- Supports **python-oracledb thin mode** (default) and optional thick mode comments.
- Uses parameterized SQL where possible and closes resources with context managers.


## 1) Dependencies

Run once in your active Jupyter kernel environment:

```bash
pip install oracledb sqlalchemy pandas matplotlib seaborn
```


In [ ]:
import os
from getpass import getpass

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import oracledb
from sqlalchemy import create_engine, text

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2) Connection settings

> In real environments, set these in shell or Jupyter secrets:
> - `ORA_USER`
> - `ORA_PASSWORD`
> - `ORA_HOST`
> - `ORA_PORT`
> - `ORA_SERVICE`


In [ ]:
# Connection metadata
CONNECTION_NAME = os.getenv("ORA_CONN_NAME", "DIPRODU")

# Defaults reflect your simulation target; override with env vars in production
ORA_USER = os.getenv("ORA_USER", "MANAMUSA")
ORA_HOST = os.getenv("ORA_HOST", "10.213.207.21")
ORA_PORT = int(os.getenv("ORA_PORT", "1521"))
ORA_SERVICE = os.getenv("ORA_SERVICE", "stdiprodu.internal.altconsult.com")

# Never hardcode default password in enterprise notebooks
ORA_PASSWORD = os.getenv("ORA_PASSWORD")
if not ORA_PASSWORD:
    ORA_PASSWORD = getpass("Enter Oracle password for MANAMUSA: ")

print(f"Profile: {CONNECTION_NAME}")
print(f"Target: {ORA_HOST}:{ORA_PORT}/{ORA_SERVICE}")

## 3) Create engine and test connection

If your org requires Oracle Client libraries (thick mode), initialize once with:

```python
# oracledb.init_oracle_client(lib_dir='/path/to/instantclient')
```


In [ ]:
dsn = oracledb.makedsn(
    host=ORA_HOST,
    port=ORA_PORT,
    service_name=ORA_SERVICE
)

engine = create_engine(
    f"oracle+oracledb://{ORA_USER}:{ORA_PASSWORD}@{dsn}",
    pool_pre_ping=True
)

with engine.connect() as conn:
    row = conn.execute(text("SELECT SYS_CONTEXT('USERENV','DB_NAME') AS db_name, CURRENT_DATE AS db_date FROM dual")).mappings().one()
    print("Connected to DB:", row["DB_NAME"])
    print("Database current date:", row["DB_DATE"])

## 4) SQL extraction from `telecom_churn`

In [ ]:
# Parameterized row limit for safer tuning
ROW_LIMIT = 5000

sql_extract = text("""
SELECT
    customer_id,
    tenure_months,
    monthly_charges,
    total_charges,
    contract_type,
    internet_service,
    payment_method,
    churn_flag
FROM telecom_churn
FETCH FIRST :row_limit ROWS ONLY
""")

with engine.connect() as conn:
    churn_df = pd.read_sql(sql_extract, conn, params={"row_limit": ROW_LIMIT})

print("Rows, columns:", churn_df.shape)
churn_df.head()

## 5) SQL business summary (churn by contract)

In [ ]:
sql_summary = text("""
SELECT
    contract_type,
    COUNT(*) AS customers,
    SUM(CASE WHEN churn_flag = 1 THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(100 * AVG(CASE WHEN churn_flag = 1 THEN 1 ELSE 0 END), 2) AS churn_rate_pct
FROM telecom_churn
GROUP BY contract_type
ORDER BY churn_rate_pct DESC
""")

with engine.connect() as conn:
    churn_by_contract = pd.read_sql(sql_summary, conn)

churn_by_contract

## 6) Python post-processing

In [ ]:
# Basic type cleanup
for col in ["tenure_months", "monthly_charges", "total_charges", "churn_flag"]:
    churn_df[col] = pd.to_numeric(churn_df[col], errors="coerce")

churn_df["churn_flag"] = churn_df["churn_flag"].fillna(0).astype(int)
churn_df["avg_charge_per_month"] = churn_df["total_charges"] / churn_df["tenure_months"].replace(0, pd.NA)

# Quick profiling view
churn_df.describe(include="all").transpose().head(20)

## 7) Visualization

In [ ]:
# 7.1 Churn rate by contract type
plt.figure()
sns.barplot(data=churn_by_contract, x="contract_type", y="churn_rate_pct", hue="contract_type", legend=False, palette="viridis")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Monthly charges by churn flag
plt.figure()
sns.boxplot(data=churn_df, x="churn_flag", y="monthly_charges")
plt.title("Monthly Charges by Churn Flag")
plt.xlabel("Churn Flag (0 = retained, 1 = churned)")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Tenure distribution split by churn
plt.figure()
sns.histplot(data=churn_df, x="tenure_months", hue="churn_flag", bins=30, kde=True, element="step")
plt.title("Tenure Distribution by Churn")
plt.xlabel("Tenure (months)")
plt.ylabel("Customer count")
plt.tight_layout()
plt.show()

## 8) Optional: save analysis artifacts

In [ ]:
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

churn_df.to_csv(f"{output_dir}/telecom_churn_extract_sample.csv", index=False)
churn_by_contract.to_csv(f"{output_dir}/telecom_churn_contract_summary.csv", index=False)

print("Saved CSV outputs to ./outputs")

## 9) Cleanup

In [ ]:
engine.dispose()
print("Oracle engine disposed cleanly.")

## 10) Troubleshooting checklist

- `ORA-12154`: verify service name / tns alias and network DNS resolution.
- `ORA-12514`: listener does not know service; confirm exact `service_name`.
- `DPI-1047`: client-library issue; use thin mode or install/configure Instant Client for thick mode.
- Permission errors on table: ask DBA for `SELECT` grant on `telecom_churn`.


## 11) Download this notebook

In JupyterLab, right-click `oracle_telecom_churn_workflow.ipynb` in the file browser and choose **Download** to save a local copy.